In [ ]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic 
%pip install litellm

In [8]:
import os
import requests
from crewai import Agent, Task, Crew
from crewai.llm import LLM
from crewai.tools import BaseTool
from dotenv import load_dotenv


In [6]:
load_dotenv()

True

In [14]:
# -------------------------------------------------- 
# Tavily Web Search Tool 
# -------------------------------------------------- 
class TavilySearchTool(BaseTool): 
    name: str = "web_search" 
    description: str = "Search the web for recent information." 
 
    def _run(self, query: str): 
        url = "https://api.tavily.com/search" 
        payload = { 
            "api_key": os.getenv("TAVILY_API_KEY"), 
            "query": query, 
            "max_results": 5 
        } 
 
        response = requests.post(url, json=payload, timeout=30) 
        response.raise_for_status() 
        data = response.json() 
 
        results = [] 
        for r in data.get("results", []): 
            title = r.get("title", "No title") 
            link = r.get("url", "No URL") 
            results.append(f"{title} - {link}") 
 
        return "\n".join(results) if results else "No web results found." 
 
 
search_tool = TavilySearchTool() 
 
# -------------------------------------------------- 
# Azure GPT-5-mini LLM via CrewAI LiteLLM path 
# -------------------------------------------------- 
llm = LLM( 
   model="gpt-5-mini",
   api_key=os.getenv("OPENAI_API_KEY"),
   is_litellm=True, 
   temperature=0.3, 
)

In [15]:
# --------------------------------------------------
# Agents
# --------------------------------------------------
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for healthcare",
    backstory=(
        "You are an expert in artificial intelligence and stay updated "
        "with the latest research trends in healthcare."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)

writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [16]:
# --------------------------------------------------
# SERIAL EXECUTION
# --------------------------------------------------
task_research = Task(
    description=(
        "Use the web search results to explain the top 3 recent advancements in AI for healthcare in 5-6 sentences."
        "Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    agent=researcher
)

task_write = Task(
    description=(
        "Write a short executive summary using the research notes provided by the AI Researcher. "
        "Limit the answer to about 200 words."
    ),
    expected_output=(
        "An executive summary report of the top 3 AI advancements in healthcare."
    ),
    agent=writer,
    context=[task_research]
)

print("\n=== SERIAL EXECUTION ===")

crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

serial_result = await crew_serial.kickoff_async()

print("\n[Serial Result]:\n")
try:
    print(serial_result.raw)
except AttributeError:
    print(serial_result)


=== SERIAL EXECUTION ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: de484141-e9da-4838-bc58-ee88774484a8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 5-6          │
│  sentences.Do not call tools again after getting results.                                                       │
│  ID: b1c2cca9-9e7f-4bb7-8c81-b4bc9583d352                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 5-6          │
│  sentences.Do not call tools again after getting results.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#14) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': "recent advancements AI healthcare 2024 2025 top advancements large language models medical    │
│  imaging protein folding federated learning FDA approvals 2024 2025 'AI in healthcare' 'latest' '20...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Executive Summary — Top 3 AI Advancements in Healthcare                                                        │
│                                                                                                                 │
│  1) Generative LLMs for Clinical Workflows                                                                      │
│  Recent large language models (Med‑PaLM 2, GPT‑4–class and multimodal variants) are being embedded into         │
│  clinical workflows to summarize notes, draft discharge summaries, support diagnostic reasoning and automate    │
│  coding. Early deployments reduce documentation burden, speed triage and improve patient communication.         │
│  Priority issues for executives: rigorous safety/provenance standards, prospective validation, and clear        │
│  governance for clinical use.                                                                                   │
│                                                                                                                 │
│  2) AI-enhanced Medical Imaging & Multimodal Diagnostics                                                        │
│  Advanced convolutional and transformer imaging models, combined with multimodal fusion (image + text + labs),  │
│  have raised sensitivity and specificity for diagnoses from stroke to cancer and accelerated triage             │
│  turnaround. Clinically cleared tools are increasingly integrated into PACS and radiology workflows as          │
│  decision-support. Implementation focus: workflow integration, radiologist oversight, and continuous            │
│  performance monitoring across populations.                                                                     │
│                                                                                                                 │
│  3) AI-driven Protein Folding & Generative Chemistry                                                            │
│  Accurate structure prediction (AlphaFold-era) plus generative molecular and foundation models enable rapid     │
│  target‑to‑lead cycles, large-scale virtual screening and de novo molecule design. These capabilities shorten   │
│  preclinical timelines and cut experimental iterations, reshaping early drug discovery. Strategic actions:      │
│  collaboration with AI-native biotech, investment in in‑silico validation, and attention to IP and regulatory   │
│  pathways.                                                                                                      │
│                                                                                                                 │
│  Recommendation: Prioritize validated pilots, cross‑functional governance, and measurable clinical and          │
│  operational KPIs before widescale rollout.                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short executive summary using the research notes provided by the AI Researcher. Limit the        │
│  answer to about 200 words.                                                                                     │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: de484141-e9da-4838-bc58-ee88774484a8                                                                       │
│  Final Output: Executive Summary — Top 3 AI Advancements in Healthcare                                          │
│                                                                                                                 │
│  1) Generative LLMs for Clinical Workflows                                                                      │
│  Recent large language models (Med‑PaLM 2, GPT‑4–class and multimodal variants) are being embedded into         │
│  clinical workflows to summarize notes, draft discharge summaries, support diagnostic reasoning and automate    │
│  coding. Early deployments reduce documentation burden, speed triage and improve patient communication.         │
│  Priority issues for executives: rigorous safety/provenance standards, prospective validation, and clear        │
│  governance for clinical use.                                                                                   │
│                                                                                                                 │
│  2) AI-enhanced Medical Imaging & Multimodal Diagnostics                                                        │
│  Advanced convolutional and transformer imaging models, combined with multimodal fusion (image + text + labs),  │
│  have raised sensitivity and specificity for diagnoses from stroke to cancer and accelerated triage             │
│  turnaround. Clinically cleared tools are increasingly integrated into PACS and radiology workflows as          │
│  decision-support. Implementation focus: workflow integration, radiologist oversight, and continuous            │
│  performance monitoring across populations.                                                                     │
│                                                                                                                 │
│  3) AI-driven Protein Folding & Generative Chemistry                                                            │
│  Accurate structure prediction (AlphaFold-era) plus generative molecular and foundation models enable rapid     │
│  target‑to‑lead cycles, large-scale virtual screening and de novo molecule design. These capabilities shorten   │
│  preclinical timelines and cut experimental iterations, reshaping early drug discovery. Strategic actions:      │
│  collaboration with AI-native biotech, investment in in‑silico validation, and attention to IP and regulatory   │
│  pathways.                                                                                                      │
│                                                                                                                 │
│  Recommendation: Prioritize validated pilots, cross‑functional governance, and measurable clinical and          │
│  operational KPIs before widescale rollout.                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[Serial Result]:

Executive Summary — Top 3 AI Advancements in Healthcare

1) Generative LLMs for Clinical Workflows
Recent large language models (Med‑PaLM 2, GPT‑4–class and multimodal variants) are being embedded into clinical workflows to summarize notes, draft discharge summaries, support diagnostic reasoning and automate coding. Early deployments reduce documentation burden, speed triage and improve patient communication. Priority issues for executives: rigorous safety/provenance standards, prospective validation, and clear governance for clinical use.

2) AI-enhanced Medical Imaging & Multimodal Diagnostics
Advanced convolutional and transformer imaging models, combined with multimodal fusion (image + text + labs), have raised sensitivity and specificity for diagnoses from stroke to cancer and accelerated triage turnaround. Clinically cleared tools are increasingly integrated into PACS and radiology workflows as decision-support. Implementation focus: workflow integration, radio

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
# --------------------------------------------------
# PARALLEL EXECUTION
# --------------------------------------------------
task_parallel_1 = Task(
    description=(
        "Use web search to list 5 AI companies focusing on drug discovery. "
        "For each company, give one short line about what they specialize in."
    ),
    expected_output="Company names and what they specialize in.",
    async_execution=True,
    agent=researcher
)

task_parallel_2 = Task(
    description=(
        "Write a short report on how AI is transforming patient diagnostics. "
        "Limit the answer to about 200 words."
    ),
    expected_output="A short report with examples and explanation.",
    agent=writer
)

print("\n=== PARALLEL EXECUTION ===")

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

parallel_result = await crew_parallel.kickoff_async()

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)


=== PARALLEL EXECUTION ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2dac7cfe-d7e0-4bb9-b251-27336d037370                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│  ID: 4c0f4a07-7e66-46ea-bfa9-efd1218ff58f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#15) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'top AI drug discovery companies 2024 list'}                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Executive report — How AI is transforming patient diagnostics                                                  │
│                                                                                                                 │
│  AI is shifting diagnostics from symptom-driven, population average approaches to rapid, mechanism-based, and   │
│  patient-specific detection. Machine learning extracts diagnostic signatures from high-dimensional data         │
│  (images, genomes, proteomes), accelerating detection, stratification, and companion-test development.          │
│  Examples from industry show concrete pathways:                                                                 │
│                                                                                                                 │
│  - Recursion Pharmaceuticals applies high-throughput image-based phenomics with ML to identify cellular         │
│  phenotype signatures—enabling automated disease classification and patient stratification from microscopy and  │
│  histology.                                                                                                     │
│  - Deep Genomics uses AI to predict the functional impact of genetic variants and to prioritize RNA-targeting   │
│  interventions, improving variant interpretation and genetic diagnosis.                                         │
│  - Insilico Medicine leverages generative AI for de novo target and biomarker discovery, feeding new molecular  │
│  signatures into diagnostic assay development.                                                                  │
│  - Atomwise and Exscientia, while focused on small-molecule discovery, accelerate identification and            │
│  optimization of ligands and probes that can become molecular imaging agents or companion diagnostics,          │
│  shortening the path from biomarker to clinic.                                                                  │
│                                                                                                                 │
│  Collectively, these approaches increase sensitivity, reduce time-to-diagnosis, and enable personalized         │
│  therapeutic decisions. Key challenges remain: clinical validation, regulatory approval, and bias mitigation.   │
│  With robust validation and integrated workflows, AI-driven diagnostics promise earlier detection, better risk  │
│  stratification, and more effective, tailored care.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short report on how AI is transforming patient diagnostics. Limit the answer to about 200        │
│  words.                                                                                                         │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2dac7cfe-d7e0-4bb9-b251-27336d037370                                                                       │
│  Final Output: Executive report — How AI is transforming patient diagnostics                                    │
│                                                                                                                 │
│  AI is shifting diagnostics from symptom-driven, population average approaches to rapid, mechanism-based, and   │
│  patient-specific detection. Machine learning extracts diagnostic signatures from high-dimensional data         │
│  (images, genomes, proteomes), accelerating detection, stratification, and companion-test development.          │
│  Examples from industry show concrete pathways:                                                                 │
│                                                                                                                 │
│  - Recursion Pharmaceuticals applies high-throughput image-based phenomics with ML to identify cellular         │
│  phenotype signatures—enabling automated disease classification and patient stratification from microscopy and  │
│  histology.                                                                                                     │
│  - Deep Genomics uses AI to predict the functional impact of genetic variants and to prioritize RNA-targeting   │
│  interventions, improving variant interpretation and genetic diagnosis.                                         │
│  - Insilico Medicine leverages generative AI for de novo target and biomarker discovery, feeding new molecular  │
│  signatures into diagnostic assay development.                                                                  │
│  - Atomwise and Exscientia, while focused on small-molecule discovery, accelerate identification and            │
│  optimization of ligands and probes that can become molecular imaging agents or companion diagnostics,          │
│  shortening the path from biomarker to clinic.                                                                  │
│                                                                                                                 │
│  Collectively, these approaches increase sensitivity, reduce time-to-diagnosis, and enable personalized         │
│  therapeutic decisions. Key challenges remain: clinical validation, regulatory approval, and bias mitigation.   │
│  With robust validation and integrated workflows, AI-driven diagnostics promise earlier detection, better risk  │
│  stratification, and more effective, tailored care.                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[Parallel Result]:

Executive report — How AI is transforming patient diagnostics

AI is shifting diagnostics from symptom-driven, population average approaches to rapid, mechanism-based, and patient-specific detection. Machine learning extracts diagnostic signatures from high-dimensional data (images, genomes, proteomes), accelerating detection, stratification, and companion-test development. Examples from industry show concrete pathways:

- Recursion Pharmaceuticals applies high-throughput image-based phenomics with ML to identify cellular phenotype signatures—enabling automated disease classification and patient stratification from microscopy and histology.  
- Deep Genomics uses AI to predict the functional impact of genetic variants and to prioritize RNA-targeting interventions, improving variant interpretation and genetic diagnosis.  
- Insilico Medicine leverages generative AI for de novo target and biomarker discovery, feeding new molecular signatures into diagnostic assay dev

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯